### NumPy Exercises: Supermarket Sales

#### 📋 Dataset Information

**Dataset**: Supermarket Sales  
**Source**: Attached file  

**Columns**:
- `Branch`: Store branch identifier where the transaction occurred ('A', 'B', or 'C').
- `Customer type`: Customer membership status ('Member', 'Normal').
- `Gende`r: Customer's gender ('Male' or 'Female').
- `Product line`: Product category purchased (6 categories: Electronic accessories, Fashion accessories, Food and beverages, Health and beauty, Home and lifestyle, Sports and travel).
- `Quantity`: Number of items purchased in the transaction.
- `Total`: Total amount paid by customer in dollars, including tax.
- `Date`: Transaction date in M/D/YYYY format.
- `Rating`: Customer satisfaction rating on a scale of 0.0 to 10.0 (higher is better).

In [1]:
import numpy as np

In [2]:
## NO NEED CODE HERE
# Load the CSV file
data = np.genfromtxt(
    'Data/supermarket_sales.csv',
    delimiter=',',
    skip_header=1,    # Skip header row
    dtype=str         # Load as strings (no preprocessing)
)

# Print basic info
print("="*60)
print("DATASET LOADED")
print("="*60)
print(f"Shape: {data.shape}")
print(f"  Rows: {data.shape[0]} transactions")
print(f"  Columns: {data.shape[1]}")

# Column names
columns = ['Branch', 'Customer type', 'Gender', 'Product line', 
           'Quantity', 'Total', 'Date', 'Rating']

print(f"\nColumns:")
for i, col in enumerate(columns):
    print(f"  [{i}] {col}")

DATASET LOADED
Shape: (1000, 8)
  Rows: 1000 transactions
  Columns: 8

Columns:
  [0] Branch
  [1] Customer type
  [2] Gender
  [3] Product line
  [4] Quantity
  [5] Total
  [6] Date
  [7] Rating


### Task 1: Customer Segmentation Analysis
**Which customer segment (by type and gender) generates the highest revenue per transaction, and how does their rating behavior differ from other segments?** 

**Instruction:**
- 4 customer segments: Member-Female, Member-Male, Normal-Female, Normal-Male.
- Calculate revenue per transaction for each segment.
- Calculate average rating for each segment.
- How each segment compares to the overall average?
   - Calculate deviations 
- Which segment is the "best"? (Based on revenue, rating and count)
   - Weight: 50% revenue, 30% rating, 20% count (volume) (Need normalize to [0.1])

In [3]:
#TODO: your code here

revenues = data[:, 5].astype(float)
ratings = data[:, 7].astype(float)

segments = ['Member-Female', 'Member-Male', 'Normal-Female', 'Normal-Male']

segment_stats = []

# Collect segment-level statistics
for seg in segments:
    tmp = np.char.add(np.char.add(data[:, 1], '-'), data[:, 2])
    mask = tmp == seg
    seg_revenues = revenues[mask]
    seg_ratings = ratings[mask]
    seg_count = len(seg_revenues)

    segment_stats.append({
        "segment": seg,
        "avg_revenue": np.mean(seg_revenues),
        "avg_rating": np.mean(seg_ratings),
        "count": seg_count
    })

# Convert to arrays for normalization
rev_arr = np.array([s["avg_revenue"] for s in segment_stats])
rat_arr = np.array([s["avg_rating"] for s in segment_stats])
cnt_arr = np.array([s["count"] for s in segment_stats])

# Normalize to [0,1]
rev_norm = (rev_arr - rev_arr.min()) / (rev_arr.max() - rev_arr.min())
rat_norm = (rat_arr - rat_arr.min()) / (rat_arr.max() - rat_arr.min())
cnt_norm = (cnt_arr - cnt_arr.min()) / (cnt_arr.max() - cnt_arr.min())

overall_avg_revenue = np.mean(revenues)
overall_avg_rating = np.mean(ratings)
overall_avg_count = np.mean(cnt_arr)

print("SEGMENT ANALYSIS\n")

# Compute scores
for i, seg in enumerate(segments):
    score = 0.5 * rev_norm[i] + 0.3 * rat_norm[i] + 0.2 * cnt_norm[i]
    segment_stats[i]["score"] = score

    print(f"Segment: {seg}")
    print(f"\tRevenue/Transaction: {segment_stats[i]['avg_revenue']:.2f}")
    print(f"\tAvg Rating:          {segment_stats[i]['avg_rating']:.2f}")
    print(f"\tCount:               {segment_stats[i]['count']}")

    print(f"\tDev Revenue:         {segment_stats[i]['avg_revenue'] - overall_avg_revenue:.2f}")
    print(f"\tDev Rating:          {segment_stats[i]['avg_rating'] - overall_avg_rating:.2f}")
    print(f"\tDev Count:           {segment_stats[i]['count'] - overall_avg_count:.2f}")

    print(f"\tWeighted Score:      {score:.3f}")
    print("-"*40)

# Best segment
best = max(segment_stats, key=lambda x: x["score"])

print(f"=> BEST SEGMENT: {best['segment']}  (Score = {best['score']:.3f})")



SEGMENT ANALYSIS

Segment: Member-Female
	Revenue/Transaction: 337.73
	Avg Rating:          6.94
	Count:               261
	Dev Revenue:         14.76
	Dev Rating:          -0.03
	Dev Count:           11.00
	Weighted Score:      0.702
----------------------------------------
Segment: Member-Male
	Revenue/Transaction: 316.99
	Avg Rating:          6.94
	Count:               240
	Dev Revenue:         -5.98
	Dev Rating:          -0.03
	Dev Count:           -10.00
	Weighted Score:      0.183
----------------------------------------
Segment: Normal-Female
	Revenue/Transaction: 332.23
	Avg Rating:          6.99
	Count:               240
	Dev Revenue:         9.27
	Dev Rating:          0.02
	Dev Count:           -10.00
	Weighted Score:      0.608
----------------------------------------
Segment: Normal-Male
	Revenue/Transaction: 305.05
	Avg Rating:          7.02
	Count:               259
	Dev Revenue:         -17.92
	Dev Rating:          0.05
	Dev Count:           9.00
	Weighted Score:      0.

### Task 2: Product Performance Optimization
**Which product lines are underperforming (below average sales) in which branches, and what is the revenue opportunity if they reached branch-average performance?**

**Instructions**:
1. **18 combinations**: 3 Branches × 6 Product Lines = 18
2. **Average sales per product-branch combination**
3. **Identify underperformers**: Which are below their branch average?
4. **Calculate gap**: How much below average?
5. **Revenue opportunity**: Potential gain if they reached branch average

In [4]:
#TODO: your code here

# Extract needed columns
branches = data[:, 0]
product_lines = data[:, 3]
totals = data[:, 5].astype(float)

unique_branches = np.unique(branches)
unique_products = np.unique(product_lines)

results = []  # store all 18 combinations
underperform = []  # store underperforming ones only

# Step 1–2: Compute avg sales for each branch
branch_avg = {}
for b in unique_branches:
    branch_avg[b] = np.mean(totals[branches == b])
    print(f"Branch {b} - Average Revenue = {branch_avg[b]:.2f}")

print("\nDETAILED PRODUCT–BRANCH PERFORMANCE\n" + "-"*60)

# Step 3: Compute 18 combinations
for b in unique_branches:
    for p in unique_products:

        mask = (branches == b) & (product_lines == p)
        product_revenues = totals[mask]

        if len(product_revenues) == 0:
            avg_sales = 0
        else:
            avg_sales = np.mean(product_revenues)

        deviation = avg_sales - branch_avg[b]  # negative = underperform
        revenue_gap = -deviation if deviation < 0 else 0  # only if underperform

        results.append({
            "branch": b,
            "product": p,
            "avg_sales": avg_sales,
            "branch_avg": branch_avg[b],
            "deviation": deviation,
            "rev_opportunity": revenue_gap
        })

        print(f"Branch {b} - {p}")
        print(f"\tAvg Sales:        {avg_sales:.2f}")
        print(f"\tBranch Avg:       {branch_avg[b]:.2f}")
        print(f"\tDeviation:        {deviation:.2f}")
        print(f"\tRevenue Gap:      {revenue_gap:.2f}")
        print("-"*40)

# Step 4: Collect underperformers
underperform = [r for r in results if r["deviation"] < 0]

print("\n" + "="*60)
print("UNDERPERFORMING PRODUCT LINES (Below Branch Average)")
print("="*60)

total_opportunity = 0

for r in underperform:
    print(f"{r['branch']} - {r['product']}: Gap = {r['rev_opportunity']:.2f}")
    total_opportunity += r['rev_opportunity']

print("-"*60)
print(f"\n=> TOTAL REVENUE OPPORTUNITY IF ALL REACH BRANCH AVERAGE: {total_opportunity:.2f}")


Branch A - Average Revenue = 312.35
Branch B - Average Revenue = 319.87
Branch C - Average Revenue = 337.10

DETAILED PRODUCT–BRANCH PERFORMANCE
------------------------------------------------------------
Branch A - Electronic accessories
	Avg Sales:        305.29
	Branch Avg:       312.35
	Deviation:        -7.07
	Revenue Gap:      7.07
----------------------------------------
Branch A - Fashion accessories
	Avg Sales:        320.25
	Branch Avg:       312.35
	Deviation:        7.89
	Revenue Gap:      0.00
----------------------------------------
Branch A - Food and beverages
	Avg Sales:        295.92
	Branch Avg:       312.35
	Deviation:        -16.44
	Revenue Gap:      16.44
----------------------------------------
Branch A - Health and beauty
	Avg Sales:        268.04
	Branch Avg:       312.35
	Deviation:        -44.32
	Revenue Gap:      44.32
----------------------------------------
Branch A - Home and lifestyle
	Avg Sales:        344.88
	Branch Avg:       312.35
	Deviation:      

### Task 3: High-Value Customer Identification
**What percentage of total revenue comes from the top 20% of transactions, and what are the common characteristics of these high-value transactions?**

**Instructions**:\
**1. Identify the Top 20% of Transactions by Total amount**

**2. Calculate Revenue Concentration**\
Determine what percentage of total revenue is generated by these top 20% of transactions.

**3. Profile High-Value Transaction Characteristics**\
For the top 20% transactions, analyze their common patterns across multiple dimensions. How many transaction happen: 
 - On each branches
 - On each product lines
 - On each customer types
 - On each gender

**4. Compare High-Value vs Overall Distribution**\
Calculate how the characteristics of high-value transactions differ from the overall dataset by comparing percentage distributions across branches, products, customer types, and gender. \
For example, if Branch A represents 40% of high-value transactions but only 33% of all transactions, this indicates Branch A has a premium customer base. 

**5. Calculate Average Metrics for High-Value Segment**\
Compute the average transaction amount, average quantity purchased, and average rating specifically for the top 20% group and compare these averages to the overall dataset averages. For example:
- Top 20% spend 125% MORE per transaction
- Top 20% buy 55% MORE items per transaction
- Top 20% Ratings ....
  
**6. Identify the "Golden Combination"**\
Find the most common combination of characteristics (Branch + Product + Customer Type + Gender) within the high-value segment

**7. Analyze Contribution by Percentile Groups**\
Beyond just the top 20%, how revenue is distributed across all percentile groups (top 20%, 20-40%, 40-60%, 60-80%, bottom 20%) 

In [5]:
# Extract needed columns
customer_type = data[:, 1]
gender = data[:, 2]
quantities = data[:, 4].astype(int)

n = len(totals)

# 1. Identify Top 20% by revenue
top_k = int(np.ceil(0.20 * n))
sorted_idx = np.argsort(totals)[::-1]
top_idx = sorted_idx[:top_k]
top_mask = np.zeros(n, dtype=bool)
top_mask[top_idx] = True

# 2. Revenue concentration
total_revenue = totals.sum()
top_revenue = totals[top_mask].sum()
top_revenue_pct = 100 * top_revenue / total_revenue

print(f"Total transactions: {n}")
print(f"Top 20% count: {top_k}")
print(f"Top 20% revenue share: {top_revenue_pct:.2f}%\n")

# Helper: compute counts & percentages WITHOUT Counter/pandas
def count_and_pct(values, mask=None):
    if mask is None:
        arr = values
    else:
        arr = values[mask]

    unique = np.unique(arr)
    result = {}
    total = len(arr)

    for u in unique:
        cnt = np.sum(arr == u)
        result[u] = (cnt, 100 * cnt / total)

    return result, total


# 3. Profile high-value
print("="*80)
print("\t3. HIGH VALUE TRANSACTION PROFILE")
print("="*80)

branch_top, _ = count_and_pct(branches, top_mask)
prod_top, _ = count_and_pct(product_lines, top_mask)
ctype_top, _ = count_and_pct(customer_type, top_mask)
gender_top, _ = count_and_pct(gender, top_mask)

print("By Branch:")
for k, (cnt, pct) in branch_top.items():
    print(f"  {k}: {cnt} ({pct:.2f}%)")
print()

print("By Product Line:")
for k, (cnt, pct) in prod_top.items():
    print(f"  {k}: {cnt} ({pct:.2f}%)")
print()

print("By Customer Type:")
for k, (cnt, pct) in ctype_top.items():
    print(f"  {k}: {cnt} ({pct:.2f}%)")
print()

print("By Gender:")
for k, (cnt, pct) in gender_top.items():
    print(f"  {k}: {cnt} ({pct:.2f}%)")
print()

# 4. Compare to overall dist
print("="*80)
print("\t4. HIGH VALUE vs OVERALL DISTRIBUTION")
print("="*80)

branch_all, _ = count_and_pct(branches)
prod_all, _ = count_and_pct(product_lines)
ctype_all, _ = count_and_pct(customer_type)
gender_all, _ = count_and_pct(gender)

def compare(all_dist, top_dist, title):
    print(title)
    keys = sorted(set(all_dist.keys()) | set(top_dist.keys()))
    for k in keys:
        all_cnt, all_pct = all_dist[k]
        top_cnt, top_pct = top_dist.get(k, (0, 0))
        diff = top_pct - all_pct
        print(f"  {k:20} overall = {all_pct:6.2f}% | top20 = {top_pct:6.2f}% | diff = {diff:+6.2f} pp")
    print()

compare(branch_all, branch_top, "Branch")
compare(prod_all, prod_top, "Product Line")
compare(ctype_all, ctype_top, "Customer Type")
compare(gender_all, gender_top, "Gender")

# 5. Average metrics: Top 20 vs Overall
print("="*80)
print("\t5. AVERAGE METRICS — Top20 vs Overall")
print("="*80)

def pct_increase(base, new):
    return 100 * (new/base - 1.0)

overall_amt = totals.mean()
top_amt = totals[top_mask].mean()

overall_qty = quantities.mean()
top_qty = quantities[top_mask].mean()

overall_rating = ratings.mean()
top_rating = ratings[top_mask].mean()

print(f"Avg amount (overall): {overall_amt:.2f}")
print(f"Avg amount (top20):  {top_amt:.2f}  -> {pct_increase(overall_amt, top_amt):+.2f}%")

print(f"Avg quantity (overall): {overall_qty:.2f}")
print(f"Avg quantity (top20):  {top_qty:.2f}  -> {pct_increase(overall_qty, top_qty):+.2f}%")

print(f"Avg rating (overall): {overall_rating:.2f}")
print(f"Avg rating (top20):   {top_rating:.2f}  -> {pct_increase(overall_rating, top_rating):+.2f}%")

print()

# 6. Golden Combination (most frequent)
print("="*80)
print("\t6. GOLDEN COMBINATION")
print("="*80)

tops = np.column_stack([
    branches[top_mask],
    product_lines[top_mask],
    customer_type[top_mask],
    gender[top_mask]
])

unique_combos, counts = np.unique(tops, axis=0, return_counts=True)
best_idx = np.argmax(counts)
best_combo = unique_combos[best_idx]
best_count = counts[best_idx]
best_pct = 100 * best_count / top_k

print(f"Most common combination = {tuple(best_combo)}")
print(f"Occurs {best_count} times ({best_pct:.2f}% of top 20%)\n")

# 7. Revenue contribution by percentile groups (20%)
print("="*80)
print("\t7. REVENUE CONTRIBUTION BY 20% GROUPS")
print("="*80)

group_size = int(np.ceil(n / 5))
labels = ["Top 20%", "20-40%", "40-60%", "60-80%", "Bottom 20%"]

for i in range(5):
    start = i * group_size
    end = min((i+1) * group_size, n)

    grp_idx = sorted_idx[start:end]
    grp_rev = totals[grp_idx].sum()
    grp_pct = 100 * grp_rev / total_revenue

    print(f"{labels[i]:10} | Txns = {len(grp_idx):3d} | Revenue = {grp_rev:10.2f} | Share = {grp_pct:6.2f}%")


Total transactions: 1000
Top 20% count: 200
Top 20% revenue share: 45.08%

	3. HIGH VALUE TRANSACTION PROFILE
By Branch:
  A: 60 (30.00%)
  B: 67 (33.50%)
  C: 73 (36.50%)

By Product Line:
  Electronic accessories: 37 (18.50%)
  Fashion accessories: 31 (15.50%)
  Food and beverages: 31 (15.50%)
  Health and beauty: 32 (16.00%)
  Home and lifestyle: 33 (16.50%)
  Sports and travel: 36 (18.00%)

By Customer Type:
  Member: 104 (52.00%)
  Normal: 96 (48.00%)

By Gender:
  Female: 104 (52.00%)
  Male: 96 (48.00%)

	4. HIGH VALUE vs OVERALL DISTRIBUTION
Branch
  A                    overall =  34.00% | top20 =  30.00% | diff =  -4.00 pp
  B                    overall =  33.20% | top20 =  33.50% | diff =  +0.30 pp
  C                    overall =  32.80% | top20 =  36.50% | diff =  +3.70 pp

Product Line
  Electronic accessories overall =  17.00% | top20 =  18.50% | diff =  +1.50 pp
  Fashion accessories  overall =  17.80% | top20 =  15.50% | diff =  -2.30 pp
  Food and beverages   overall 